### **TIDIGITS**

In [4]:
import numpy as np
from library.gmm import GMM
from library.knn import KNN
from library.lda import LDA
from library.pca import PCA

### **Load Data**

In [5]:
sequences = []
labels = []
for digit in [2, 4, 7]:
    with np.load(f"./data/tidigits/{digit}.npz") as data:
        for key in data.files:
            sequences.append(data[key])
            labels.append(digit)

X = []
for sequence in sequences:
    parts = np.array_split(sequence, 4)
    means = [part.mean(axis=0) for part in parts]
    X.append(np.concatenate(means))
X = np.array(X)
y = np.array(labels)

### **Shuffle and Split**

In [6]:
rng = np.random.default_rng(42)
indices = rng.permutation(len(y))
train_end = int(0.6 * len(y))
val_end = int(0.8 * len(y))

train_indices = indices[:train_end]
val_indices = indices[train_end:val_end]
test_indices = indices[val_end:]

X_train = X[train_indices]
y_train = y[train_indices]
X_val = X[val_indices]
y_val = y[val_indices]
X_test = X[test_indices]
y_test = y[test_indices]

classes = np.unique(y_train)
print("Class order:", classes)
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Class order: [2 4 7]
Train: (810, 156)
Validation: (270, 156)
Test: (270, 156)


### **GMM**

In [7]:
models = []
priors = []
for label in classes:
    samples = X_train[y_train == label]
    model = GMM(n_components=3, regularization=1e-3, iterations=100, convergence_tol=1e-4)
    model.fit(samples)
    models.append(model)
    priors.append(len(samples) / len(X_train))
    print("Class", label, "EM iterations:", len(model.log_likelihoods))
    if len(model.log_likelihoods) == model.iterations:
        print("Iteration limit reached; convergence is not guaranteed.")

val_scores = np.column_stack([model.score_samples(X_val) for model in models])
gmm_val_predictions = classes[np.argmax(val_scores + np.log(priors), axis=1)]

test_scores = np.column_stack([model.score_samples(X_test) for model in models])
gmm_test_predictions = classes[np.argmax(test_scores + np.log(priors), axis=1)]

# Confusion matrix
gmm_confusion = np.zeros((len(classes), len(classes)), dtype=int)
for i, actual in enumerate(classes):
    for j, predicted in enumerate(classes):
        gmm_confusion[i, j] = np.sum(
            (y_test == actual) & (gmm_test_predictions == predicted))

print("Validation accuracy:", np.mean(gmm_val_predictions == y_val))
print("Test accuracy:", np.mean(gmm_test_predictions == y_test))
print("Test confusion matrix:\n", gmm_confusion)

Class 2 EM iterations: 2
Class 4 EM iterations: 2
Class 7 EM iterations: 2
Validation accuracy: 0.937037037037037
Test accuracy: 0.937037037037037
Test confusion matrix:
 [[77  3  8]
 [ 2 90  2]
 [ 2  0 86]]


### **KNN**

In [8]:
knn = KNN(neighbors=5)
knn.fit(X_train, y_train)
knn_val_predictions = knn.predict(X_val)
knn_test_predictions = knn.predict(X_test)

# Confusion matrix
knn_confusion = np.zeros((len(classes), len(classes)), dtype=int)
for i, actual in enumerate(classes):
    for j, predicted in enumerate(classes):
        knn_confusion[i, j] = np.sum(
            (y_test == actual) & (knn_test_predictions == predicted))

print("Validation accuracy:", np.mean(knn_val_predictions == y_val))
print("Test accuracy:", np.mean(knn_test_predictions == y_test))
print("Test confusion matrix:\n", knn_confusion)

Validation accuracy: 0.9814814814814815
Test accuracy: 0.9925925925925926
Test confusion matrix:
 [[87  1  0]
 [ 1 93  0]
 [ 0  0 88]]


### **LDA**

In [9]:
lda = LDA()
lda.fit(X_train, y_train)
lda_val_predictions = lda.predict(X_val)
lda_test_predictions = lda.predict(X_test)

# Confusion matrix
lda_confusion = np.zeros((len(classes), len(classes)), dtype=int)
for i, actual in enumerate(classes):
    for j, predicted in enumerate(classes):
        lda_confusion[i, j] = np.sum(
            (y_test == actual) & (lda_test_predictions == predicted))

print("Validation accuracy:", np.mean(lda_val_predictions == y_val))
print("Test accuracy:", np.mean(lda_test_predictions == y_test))
print("Test confusion matrix:\n", lda_confusion)

Validation accuracy: 0.9703703703703703
Test accuracy: 0.9888888888888889
Test confusion matrix:
 [[87  0  1]
 [ 1 92  1]
 [ 0  0 88]]


### **PCA**

In [10]:
K = 39
pca = PCA(n_components=K)
pca.fit(X_train)
X_train_pca = pca.transform(X_train)
X_val_pca = pca.transform(X_val)
X_test_pca = pca.transform(X_test)

print("Train:", X_train_pca.shape)
print("Validation:", X_val_pca.shape)
print("Test:", X_test_pca.shape)

Train: (810, 39)
Validation: (270, 39)
Test: (270, 39)


### **GMM with PCA**

In [11]:
pca_models = []
pca_priors = []
for label in classes:
    pca_samples = X_train_pca[y_train == label]
    pca_model = GMM(n_components=3, regularization=1e-3, iterations=100, convergence_tol=1e-4)
    pca_model.fit(pca_samples)
    pca_models.append(pca_model)
    pca_priors.append(len(pca_samples) / len(X_train_pca))
    print("Class", label, "EM iterations:", len(pca_model.log_likelihoods))
    if len(pca_model.log_likelihoods) == pca_model.iterations:
        print("Iteration limit reached; convergence is not guaranteed.")

pca_val_scores = np.column_stack([pca_model.score_samples(X_val_pca) for pca_model in pca_models])
gmm_pca_val_predictions = classes[np.argmax(pca_val_scores + np.log(pca_priors), axis=1)]

pca_test_scores = np.column_stack([pca_model.score_samples(X_test_pca) for pca_model in pca_models])
gmm_pca_test_predictions = classes[np.argmax(pca_test_scores + np.log(pca_priors), axis=1)]

# Confusion matrix
gmm_pca_confusion = np.zeros((len(classes), len(classes)), dtype=int)
for i, actual in enumerate(classes):
    for j, predicted in enumerate(classes):
        gmm_pca_confusion[i, j] = np.sum(
            (y_test == actual) & (gmm_pca_test_predictions == predicted))

print("Validation accuracy:", np.mean(gmm_pca_val_predictions == y_val))
print("Test accuracy:", np.mean(gmm_pca_test_predictions == y_test))
print("Test confusion matrix:\n", gmm_pca_confusion)

Class 2 EM iterations: 5
Class 4 EM iterations: 4
Class 7 EM iterations: 4
Validation accuracy: 0.9925925925925926
Test accuracy: 1.0
Test confusion matrix:
 [[88  0  0]
 [ 0 94  0]
 [ 0  0 88]]


### **KNN with PCA**

In [12]:
knn_pca = KNN(neighbors=5)
knn_pca.fit(X_train_pca, y_train)
knn_pca_val_predictions = knn_pca.predict(X_val_pca)
knn_pca_test_predictions = knn_pca.predict(X_test_pca)

# Confusion matrix
knn_pca_confusion = np.zeros((len(classes), len(classes)), dtype=int)
for i, actual in enumerate(classes):
    for j, predicted in enumerate(classes):
        knn_pca_confusion[i, j] = np.sum(
            (y_test == actual) & (knn_pca_test_predictions == predicted))

print("Validation accuracy:", np.mean(knn_pca_val_predictions == y_val))
print("Test accuracy:", np.mean(knn_pca_test_predictions == y_test))
print("Test confusion matrix:\n", knn_pca_confusion)

Validation accuracy: 0.9777777777777777
Test accuracy: 0.9925925925925926
Test confusion matrix:
 [[87  1  0]
 [ 1 93  0]
 [ 0  0 88]]


### **LDA with PCA**

In [13]:
lda_pca = LDA()
lda_pca.fit(X_train_pca, y_train)
lda_pca_val_predictions = lda_pca.predict(X_val_pca)
lda_pca_test_predictions = lda_pca.predict(X_test_pca)

# Confusion matrix
lda_pca_confusion = np.zeros((len(classes), len(classes)), dtype=int)
for i, actual in enumerate(classes):
    for j, predicted in enumerate(classes):
        lda_pca_confusion[i, j] = np.sum(
            (y_test == actual) & (lda_pca_test_predictions == predicted))

print("Validation accuracy:", np.mean(lda_pca_val_predictions == y_val))
print("Test accuracy:", np.mean(lda_pca_test_predictions == y_test))
print("Test confusion matrix:\n", lda_pca_confusion)

Validation accuracy: 0.9703703703703703
Test accuracy: 0.9703703703703703
Test confusion matrix:
 [[85  0  3]
 [ 1 91  2]
 [ 1  1 86]]
